In [3]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5674.08it/s]


In [10]:
import numpy as np

In [11]:
import faiss

In [4]:
knowledge_base = [
    "To reset your password, click on the forgot password link on the login page.",
    "Users can update their billing information from the account settings page.",
    "If your account is locked, contact customer support for assistance.",
    "You can change your registered email address under profile settings.",
    "Payment failures may occur due to insufficient account balance.",
    "Two-factor authentication improves account security.",
    "Login issues can be resolved by clearing browser cookies and cache.",
    "Refund requests are processed within five business days.",
    "Subscription plans can be upgraded from the billing dashboard.",
    "Users can delete their account permanently from privacy settings."
]

In [5]:
embeddings = model.encode(
    knowledge_base,
    convert_to_numpy=True
)

In [6]:
print(embeddings.shape)


(10, 384)


In [7]:
dimension = embeddings.shape[1]


In [12]:
faiss.normalize_L2(embeddings)


In [13]:
index = faiss.IndexFlatL2(dimension)


In [14]:
index.add(embeddings)


In [15]:
print("\nTotal vectors stored:")
print(index.ntotal)


Total vectors stored:
10


In [16]:


def semantic_search(query, top_k=3):
    query_embedding = model.encode(
        [query],
        convert_to_numpy=True
    )

    faiss.normalize_L2(query_embedding)

    distances, indices = index.search(
        query_embedding,
        top_k
    )

    print("\nQuery:", query)
    print("-" * 80)
    print(f"{'Rank':<6}{'Score':<15}{'Matched Sentence'}")
    print("-" * 80)

    for rank, (idx, score) in enumerate(
        zip(indices[0], distances[0]),
        start=1
    ):
        print(
            f"{rank:<6}{score:<15.4f}{knowledge_base[idx]}"
        )



In [17]:


test_queries = [
    "I forgot my password",
    "How can I update payment details?",
    "Unable to login to my account"
]


for query in test_queries:
    semantic_search(query)

while True:
    user_query = input("\nAsk a question (or type exit): ")

    if user_query.lower() == "exit":
        print("Goodbye!")
        break

    semantic_search(user_query)


Query: I forgot my password
--------------------------------------------------------------------------------
Rank  Score          Matched Sentence
--------------------------------------------------------------------------------
1     0.5466         To reset your password, click on the forgot password link on the login page.
2     0.8577         If your account is locked, contact customer support for assistance.
3     1.3685         Two-factor authentication improves account security.

Query: How can I update payment details?
--------------------------------------------------------------------------------
Rank  Score          Matched Sentence
--------------------------------------------------------------------------------
1     0.7868         Users can update their billing information from the account settings page.
2     1.1657         Subscription plans can be upgraded from the billing dashboard.
3     1.2699         If your account is locked, contact customer support for assistance.